# Azure OpenAI (APIM) Chat Demo

This notebook is a walkthrough version of `Lab2.py`. It shows how to call an **Azure OpenAI**
model through an **Azure API Management (APIM)** gateway using the standard `openai` Python
SDK, and it repeats the same conversational flow four times to demonstrate a simple
"ask → answer → self-evaluate" pattern:

1. Ask the model for a fun fact.
2. Ask the model to invent a hard, IQ-style question.
3. Ask the model to answer its own question.
4. Ask the model to evaluate whether that answer was correct.

Each code cell below is preceded by an explanation of what it does and why.

## 1. Imports

- `dotenv.load_dotenv` reads key/value pairs from a local `.env` file into environment variables.
- `AzureOpenAI` is the Azure-flavored client class from the `openai` package (as opposed to the
  plain `OpenAI` class used for api.openai.com).

In [ ]:
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI

## 2. Load environment variables

`load_dotenv(override=True)` looks for a `.env` file in the current directory and loads its
contents into `os.environ`. `override=True` means values in `.env` take priority over any values
already set in the shell environment.

Make sure you have a `.env` file (in the same folder as this notebook, or on `sys.path`) that
defines:

```
AZURE_APIM_OPENAI_SUBSCRIPTION_KEY=...
AZURE_APIM_OPENAI_API_VERSION=...
AZURE_APIM_OPENAI_ENDPOINT=https://<your-apim-instance>.azure-api.net
AZURE_APIM_OPENAI_DEPLOYMENT=<your-deployment-name>
```

**Note on the endpoint:** it should be the *host only* — e.g.
`https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net` — **without** a trailing
`/openai/deployments/...` path. The SDK builds the full path itself using the API version and
deployment name.

In [ ]:
# Read .env and override any existing process env values.
load_dotenv(override=True)

# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")

## 3. Validate configuration

Before making any network calls, confirm that all four required settings were actually found.
`all([...])` returns `False` if any of them is `None` or an empty string, in which case
`sys.exit(...)` prints a helpful message and stops execution (raises `SystemExit` — in a notebook
this will show as an error, which is expected if `.env` is missing or incomplete).

If configuration is present, we print a masked preview of the API key (first 8 characters only)
and the deployment name, just to confirm — without ever logging the key can be verified without
leaking the whole secret.

In [ ]:
if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")

## 4. Create the Azure OpenAI client

`AzureOpenAI` is instantiated once and reused for every request. Note the three arguments it
needs, which differ from the plain `OpenAI` client:

- `api_key` — here it's actually the APIM **subscription key**, not an Azure OpenAI resource key,
  since requests are routed through the APIM gateway.
- `api_version` — the Azure OpenAI REST API version (e.g. `2024-06-01`), required because Azure
  versions its API explicitly, unlike api.openai.com.
- `azure_endpoint` — the base host of the APIM instance (see the note above).

In [ ]:
# Sync client pointed at Azure APIM.
openai = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

## 5. A small `chat()` helper

This wraps `openai.chat.completions.create(...)` so the rest of the notebook doesn't repeat
boilerplate. Two Azure-specific details are worth calling out:

- **`model=deployment`** — On Azure, the `model` parameter is actually the *deployment name* you
  configured in the Azure OpenAI resource (e.g. `gpt-5-chat`), not a generic model id like
  `gpt-5`. Azure routes the request based on that deployment.
- **`max_completion_tokens` instead of `max_tokens`** — Newer reasoning-capable deployments
  (GPT-5-style) reject the older `max_tokens` parameter and require `max_completion_tokens`
  instead. The helper defaults this to `300` but lets callers override it (used later with `1000` for the longer evaluation step).

In [ ]:
# On Azure the `model` argument is the *deployment name*, not an OpenAI model id.
# GPT-5 deployments need max_completion_tokens (max_tokens is rejected).
def chat(messages, max_completion_tokens=1000):
    return openai.chat.completions.create(
        model=deployment,
        messages=messages,
        max_completion_tokens=max_completion_tokens,
    )

## 6. Step 1 — Ask for a fun fact

The simplest possible call: a single user message, default token limit (300), and we print the
model's reply text (`response.choices[0].message.content`).

In [ ]:
# 1) Fun fact
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
print(response.choices[0].message.content)

## 7. Step 2 — Ask the model to invent a hard question

We prompt the model to generate a challenging, IQ-style question, instructing it to respond
**only** with the question itself (no preamble) so that `question` can be reused directly as the
next prompt.

In [ ]:
# 2) Ask the model to invent a hard IQ-style question
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)

## 8. Step 3 — Ask the model to answer its own question

The `question` text generated in the previous step is now sent back to the model as a fresh
prompt (a brand-new `messages` list — the model has no memory of generating the question; it's
just answering it as if seeing it cold).

Since we're already in a notebook, we can render the answer as nicely formatted Markdown using
`IPython.display` directly — no fallback needed.

In [ ]:
# 3) Ask the model to answer that question
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)

In [ ]:
# Render the answer as Markdown in the notebook.
from IPython.display import Markdown, display

display(Markdown(answer))

## 9. Step 4 — Ask the model to evaluate the answer

Finally, we build a single prompt string that includes both the `question` and the `answer`
(using an f-string), and ask the model to judge whether the answer is correct. This is a common
"LLM-as-judge" pattern: use the same (or another) model to self-critique its own prior output.

This call uses a higher `max_completion_tokens` (5000 instead of the default 300) since an
evaluation with reasoning tends to need more room than a short fun fact.

In [ ]:
# 4) Ask the model to evaluate the answer
deployment="gpt-5.4-ptu"
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)

In [ ]:
messages = [{"role": "user", "content": message}]
response = chat(messages, max_completion_tokens=5000)
print(response.choices[0].message.content)

## Summary

This notebook demonstrated:

- Loading Azure APIM credentials safely from a `.env` file.
- Configuring an `AzureOpenAI` client to route through an APIM gateway.
- The Azure-specific quirks (`model` = deployment name, `max_completion_tokens` vs `max_tokens`).
- A four-step "generate → answer → self-evaluate" prompt chain, showing how each model response
  can feed into the next request.

To adapt this for your own use, edit the prompts in each step, or swap the `chat()` helper's
`max_completion_tokens` values to fit your deployment's needs.